# Advent of Code 2025

## Puzzle 1

### Part One

In [25]:
from dataclasses import dataclass
from pathlib import Path
from typing import Literal


@dataclass
class Move:
    dir: Literal["L", "R"]
    dist: int


def iter_moves():
    file = Path().parent / "inputs" / "01.txt"
    text = file.read_text()
    for line in text.splitlines():
        dir = line[0]
        dist = int(line[1:])
        assert dir == "L" or dir == "R"
        yield Move(dir, dist)


num_zeroes = 0
pos = 50
for move in iter_moves():
    if move.dir == "L":
        pos = (pos - move.dist) % 100
    elif move.dir == "R":
        pos = (pos + move.dist) % 100
    else:
        raise ValueError("Invalid move direction")
    if pos == 0:
        num_zeroes += 1

print(f"Number of times dial stopped on zero: {num_zeroes}")

Number of times dial stopped on zero: 1066


### Part Two

In [27]:
num_zero_clicks = 0
pos = 50
for move in iter_moves():
    if move.dir == "L":
        # => 100 not possible
        # 1..99 -> 0
        # -99..0 -> 1
        # -199..-100 -> 2
        num_zero_clicks += abs((pos - move.dist - 1) // 100)
        if pos == 0:
            num_zero_clicks -= 1
        pos = (pos - move.dist) % 100
    elif move.dir == "R":
        # <= 0 not possible
        # 1..99 -> 0
        # 100..199 -> 1
        # 200..299 -> 2
        num_zero_clicks += (pos + move.dist) // 100
        pos = (pos + move.dist) % 100
    else:
        raise ValueError("Invalid move direction")

print(f"Number of times dial passed zero: {num_zero_clicks}")

Number of times dial passed zero: 6223


## Puzzle 2

### Part One

In [ ]:
from pathlib import Path


def iter_ranges():
    file = Path().parent / "inputs" / "02.txt"
    text = file.read_text()
    for line in text.split(","):
        start, end = line.split("-")
        yield int(start), int(end)

(385350926, 385403705)

In [ ]:
def is_invalid_id(id: int):
    num_digits = len(str(id))
    if num_digits % 2 != 0:
        # Only IDs with an even number of digits can be invalid
        return False
    # The first half of the ID must match the second half
    return str(id)[: num_digits // 2] == str(id)[num_digits // 2 :]

In [6]:
sum_invalid_ids = 0
for start, end in iter_ranges():
    for id in range(start, end + 1):
        if is_invalid_id(id):
            sum_invalid_ids += id

print(sum_invalid_ids)

28846518423


### Part Two

In [22]:
def is_invalid_id(id: int):
    num_digits = len(str(id))  # 4
    for i in range(num_digits - 1):  # 0..2
        str_to_repeat = str(id)[: i + 1]  # 1..3
        if num_digits % len(str_to_repeat) != 0:
            continue
        num_times_to_repeat = num_digits // len(str_to_repeat)
        if str(id) == str_to_repeat * num_times_to_repeat:
            return True
    return False

In [23]:
sum_invalid_ids = 0
for start, end in iter_ranges():
    for id in range(start, end + 1):
        if is_invalid_id(id):
            sum_invalid_ids += id

print(sum_invalid_ids)

31578210022


## Puzzle 3

### Part One

In [6]:
from pathlib import Path


def iter_banks():
    file = Path().parent / "inputs" / "03.txt"
    text = file.read_text()
    for line in text.splitlines():
        yield [int(c) for c in line]


In [ ]:
def get_maximum_joltage(bank: list[int]):
    if len(bank) < 2:
        return 0

    # Find the largest battery by scanning the bank from left to right,
    # keeping the leftmost battery in case of a tie,
    # and excluding the very last battery (since we need a second battery)
    first_battery_idx = None
    first_battery_joltage = None
    for i, battery in enumerate(bank[:-1]):
        if first_battery_joltage is None or battery > first_battery_joltage:
            first_battery_idx = i
            first_battery_joltage = battery
            if battery == 9:
                break  # can't find a better battery

    assert first_battery_idx is not None
    assert first_battery_joltage is not None

    # Second battery joltage is the largest battery to the right of the
    # first battery
    second_battery_joltage = max(bank[first_battery_idx + 1 :])

    # Calculate the combined joltage
    return int(str(first_battery_joltage) + str(second_battery_joltage))

In [24]:
sum = 0
for bank in iter_banks():
    sum += get_maximum_joltage(bank)
print(sum)

17142


### Part Two

In [48]:
def get_maximum_joltage(bank: list[int], n=12):
    if len(bank) < n:
        raise ValueError("Not enough batteries in bank")
    if n < 1:
        raise ValueError("Need to choose at least one battery")
    # If we can only choose one battery, we take the largest one
    if n == 1:
        return max(bank)

    # Find the largest battery by scanning the bank from left to right,
    # keeping the leftmost battery in case of a tie,
    # and excluding the last n-1 batteries (since we need n-1 more batteries
    # after this one)
    next_battery_idx = None
    next_battery_joltage = None
    for i, battery in enumerate(bank[: -(n - 1)]):
        if next_battery_joltage is None or battery > next_battery_joltage:
            next_battery_idx = i
            next_battery_joltage = battery
            if battery == 9:
                break  # can't find a better battery

    assert next_battery_idx is not None
    assert next_battery_joltage is not None

    return int(
        str(next_battery_joltage)
        + str(get_maximum_joltage(bank[next_battery_idx + 1 :], n=n - 1))
    )


get_maximum_joltage([4, 2, 3, 9, 2, 1, 3], n=5)

49213

In [49]:
sum = 0
for bank in iter_banks():
    sum += get_maximum_joltage(bank)
print(sum)

169935154100102
